In [3]:
import numpy as np
import pandas as pd

# ----------------------------
# 1. Load connectivity data
# ----------------------------
print("Loading connectivity features...")
conn = np.load('/home/jaizor/jaizor/xtra/notebooks/UKBB/data/fMRI_numpy/correlation_connectivity_features_complete.npz')
conn_features = conn['features']               # (n_conn, n_features)
conn_eids = conn['subject_ids'].astype(str)    # Ensure string for safe merge
feature_names = conn['feature_names']
region_names = conn['region_names']

print(f"Connectivity: {conn_features.shape[0]} subjects, {conn_features.shape[1]} features")

# ----------------------------
# 2. Load phenotypic data
# ----------------------------
print("Loading phenotypic data...")
pheno = pd.read_csv('/home/jaizor/jaizor/xtra/notebooks/UKBB/data/csv/_/fMRI_final_with_health_status.csv', dtype={'eid': str})
print(f"Phenotype: {len(pheno)} subjects")

# Ensure 'eid' is string (critical for merge)
pheno['eid'] = pheno['eid'].astype(str)

# ----------------------------
# 3. Find intersection of EIDs + DEBUG LOGS
# ----------------------------
conn_set = set(conn_eids)
pheno_set = set(pheno['eid'])
common_eids = sorted(conn_set & pheno_set)  # Sorted for deterministic order

# 🔍 DEBUG: Log mismatches
only_in_conn = conn_set - pheno_set
only_in_pheno = pheno_set - conn_set

print(f"✅ {len(common_eids)} subjects in BOTH datasets.")
print(f"⚠️  {len(only_in_conn)} subjects in connectivity but MISSING in phenotype")
print(f"⚠️  {len(only_in_pheno)} subjects in phenotype but MISSING in connectivity")

if len(common_eids) == 0:
    raise ValueError("No overlapping subjects! Check EID formats.")

# Optional: Save mismatched IDs for deep inspection (uncomment if needed)
# np.savetxt('debug_only_in_connectivity.txt', sorted(only_in_conn), fmt='%s')
# np.savetxt('debug_only_in_phenotype.txt', sorted(only_in_pheno), fmt='%s')

# ----------------------------
# 4. Reorder both datasets to match common_eids
# ----------------------------
# Build index maps
conn_eid_to_idx = {eid: i for i, eid in enumerate(conn_eids)}
pheno_eid_to_idx = {eid: i for i, eid in enumerate(pheno['eid'])}

# Get indices in original arrays
conn_indices = [conn_eid_to_idx[eid] for eid in common_eids]
pheno_indices = [pheno_eid_to_idx[eid] for eid in common_eids]

# Subset and reorder
X = conn_features[conn_indices]          # (n_common, n_features)
pheno_aligned = pheno.iloc[pheno_indices].reset_index(drop=True)

# Sanity check
assert list(pheno_aligned['eid']) == common_eids, "EID mismatch after alignment!"
assert X.shape[0] == len(common_eids), "Row count mismatch!"

print(f"✅ Aligned dataset: {X.shape[0]} subjects, {X.shape[1]} connectivity features")

# ----------------------------
# 5. Save aligned data
# ----------------------------
# Save phenotypes (CSV)
pheno_aligned.to_csv('aligned_phenotypes.csv', index=False)
print("✅ Saved aligned phenotypes to 'aligned_phenotypes.csv'")

# Save connectivity + metadata (NPZ)
np.savez_compressed(
    'aligned_connectivity_features.npz',
    features=X,
    subject_ids=common_eids,
    feature_names=feature_names,
    region_names=region_names
)
print("✅ Saved aligned connectivity to 'aligned_connectivity_features.npz'")

# Save EID list
np.savetxt('aligned_subject_eids.txt', common_eids, fmt='%s')
print("✅ Saved EID list to 'aligned_subject_eids.txt'")

Loading connectivity features...
Connectivity: 16340 subjects, 85491 features
Loading phenotypic data...
Phenotype: 90629 subjects
✅ 16340 subjects in BOTH datasets.
⚠️  0 subjects in connectivity but MISSING in phenotype
⚠️  74289 subjects in phenotype but MISSING in connectivity
✅ Aligned dataset: 16340 subjects, 85491 connectivity features
✅ Saved aligned phenotypes to 'aligned_phenotypes.csv'
✅ Saved aligned connectivity to 'aligned_connectivity_features.npz'
✅ Saved EID list to 'aligned_subject_eids.txt'
